# Exercise 2: video → frames → JSON timeline

## Tasks

1. **Frames:** cut a short clip into single pictures.
2. **Describe:** let the OpenAI API describe one frame – then all of them.
3. **Listen:** turn the speech into text with start and end times.
4. **Merge:** put pictures and speech on one timeline.

**This is the solution.** Try `video_timeline.ipynb` first.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code. The API calls are in `helper.py` – open it to see how they work.

In [ ]:
# Standard Python libraries.
import sys

# Libraries installed from requirements.txt.
import pandas as pd

# helper.py is in the main folder, one level up.
sys.path.append("..")

In [ ]:
# Functions from helper.py.
from helper import DATA, describe, extract_frames, show_images, transcribe

## 1. Cut the clip into one picture per second

The frames are saved in the folder `frames/` – you can open them in the Explorer.

In [ ]:
# The video to analyse.
video_path = DATA / "videos" / "ueli_scene3_talk.mp4"

# Save one frame per second and show them.
paths = extract_frames(video_path)
show_images(paths)

## 2. Let the OpenAI API describe one frame – then all of them

While the cells run: create your Google account for Flow.

In [ ]:
# The instruction for the model.
PROMPT = """
Describe this video frame in one short sentence.
Does Ueli look toward the camera?
"""

In [ ]:
# Try it on one frame. Change the number (0 to 7) to try another one.
frame_path = paths[2]
answer = describe(PROMPT, frame_path)
print(answer)

In [ ]:
# Describe every frame. t is the time in seconds.
rows = []
for t in range(len(paths)):
    description = describe(PROMPT, paths[t])
    rows.append({"t": t, "description": description})

# One row per frame.
frames = pd.DataFrame(rows)
frames

## 3. Turn the speech into text with start and end times

`transcribe()` sends the video to Whisper, OpenAI's speech-to-text model. Swiss German often comes back as Standard German.

In [ ]:
# One row per speech segment: start, end and text.
speech = transcribe(video_path)
speech

## 4. Put pictures and speech on one timeline

A frame belongs to a speech segment when `start ≤ t < end`. The time is the key that joins what is seen and what is said.

In [ ]:
# Copy the text of each speech segment to the frames inside [start, end).
frames["said"] = ""
for segment in speech.itertuples():
    inside = frames["t"].between(segment.start, segment.end, inclusive="left")
    frames.loc[inside, "said"] = segment.text

# What is seen and what is said at the same moment.
frames

A short line between two frames is missing in this table. That is why the saved timeline keeps every frame **and** every speech segment as its own row.

In [ ]:
# One row per frame: it covers one second.
pictures = pd.DataFrame()
pictures["start"] = frames["t"]
pictures["end"] = frames["t"] + 1
pictures["kind"] = "picture"
pictures["text"] = frames["description"]

# One row per speech segment.
spoken = speech.copy()
spoken["kind"] = "speech"

# Put both tables on one timeline and sort it by time.
timeline = pd.concat([pictures, spoken])
timeline = timeline.sort_values("start")
timeline

In [ ]:
# Save the timeline as JSON lines: one event per line.
timeline.to_json("timeline.jsonl", orient="records", lines=True)

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. What do you lose with one frame every 2 seconds instead of every second? Look at «Lueg, Bruno» in the table of frames.
2. Why do we send 8 frames to the OpenAI API and not all 190 frames of the clip?
3. A frame gets the text of a speech segment when `start ≤ t < end`. Why is the end not included?
4. Compare the transcript with what Ueli says in Swiss German. What did Whisper change?
5. The model describes each frame on its own. What can it therefore not know about the video?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. With frames at 0 s and 2 s, «Lueg, Bruno» (0.5–1.5 s) lies between two frames, so no frame carries it. Short events get lost. That is why the timeline keeps every speech segment as its own row.

2. Every frame costs tokens and time, and neighbouring frames look almost the same. Too few frames miss events, too many cost money – so you have to choose.

3. A frame exactly on the border would otherwise belong to two segments. With [start, end) every point in time belongs to exactly one segment.

4. Whisper often answers in Standard German or misspells dialect words. Compare the transcript with a reference before you use it.

5. It sees no motion, no order and no sound: not what happened before or after, and not how long something lasts. This information comes from our time stamps and from the merge.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')